# Penanda linguistik gender — ETHICS

## SPEC · 4 Oktober 2026

Hitung ulang langsung dari JSON asli untuk **seluruh respons enam model dan sembilan persona**. Notebook ini menggantikan tampilan sumber lama yang memiliki masalah pairing/kanal teks; versi sebelumnya tersimpan di `notebooks/gender/_arsip/sebelum_raw_rebuild_20261004/`.

**Kandidat, bukan hasil final tesis.** Perbaikan fondasi tidak menyatakan audit parsing maupun precision/recall kamus sudah selesai. Semua respons termasuk abstain diinventarisasi; tidak ada penilaian benar/salah baru. Bab IV dan Bab VI tidak disentuh.

## DESIGN PLAN

1. Pilih kanal jawaban akhir, tanpa fallback reasoning; simpan provenance dan SHA-256 JSON.
2. Cocokkan isi soal untuk pairing; keluarkan duplikasi/soal tak terpetakan dari pasangan, bukan dari inventaris.
3. Pertahankan aturan marker yang ada, tampilkan raw count/density/presence dan tiga ukuran panjang/keragaman.
4. Hitung statistik kandidat dan PNG konsisten, lalu pasang hanya setelah assertions lolos.

Jalankan semua sel dari kernel baru. Setiap eksekusi membuat folder hasil baru; data lama di result/ tidak ditimpa.


In [ ]:
BENCHMARK = "ETHICS"
NOTEBOOK_NAME = "ETHICS_Gender_Linguistic_Markers_Analysis_v1_5_candidate.ipynb"
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import dataclasses, hashlib, html, importlib, json, re, shutil, sys, unicodedata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import friedmanchisquare
from IPython.display import Image, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "AGENTS.md").is_file() and (p / "result").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ACTIVE = ROOT / "notebooks/gender/linguistic marker" / BENCHMARK
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN = ROOT / "result/linguistic_markers" / f"gender_{BENCHMARK.lower()}_raw_rebuild_{RUN_ID}"
RUN.mkdir(parents=True, exist_ok=False)
STAGED_FIGURES = RUN / "figures"
STAGED_FIGURES.mkdir()
MODELS = [
    "meta-llama_llama-3.1-8b-instruct", "meta-llama_llama-3.3-70b-instruct",
    "mistralai_ministral-8b-2512", "deepseek_deepseek-r1-distill-qwen-32b",
    "openai_gpt-oss-20b", "qwen_qwen3-next-80b-a3b-thinking",
]
MODEL_LABELS = dict(zip(MODELS, ["Llama 3.1 8B", "Llama 3.3 70B", "Ministral 8B", "DeepSeek R1 32B", "GPT-OSS 20B", "Qwen3 Next 80B"]))
COLORS = dict(zip(MODELS, ["#2463EB", "#8038DC", "#00946A", "#D52C37", "#C26A00", "#007E99"]))
CONDITIONS = ["explicit_1", "implicit_1", "implicit_2", "implicit_3"]
CONDITION_LABELS = dict(zip(CONDITIONS, ["E", "I1", "I2", "I3"]))
PERSONAS = ["neutral"] + [f"{gender}_{c}" for gender in ("man", "woman") for c in CONDITIONS]
PERSONA_LABELS = ["N"] + [f"{gender} {CONDITION_LABELS[c]}" for gender in ("Man", "Woman") for c in CONDITIONS]
SEED = 20261004
BOOTSTRAP_RESAMPLES = 2000
SCIENTIFIC_STATUS = "candidate_not_validated_for_thesis"
# These are candidates, not a declaration that parser/dictionary audits passed.
FINAL_RELEASE = False
assert not FINAL_RELEASE, "Final release requires independently completed source/parser/marker audits."
gsm_extractor = importlib.import_module("src.MAIN_PARSING.Linguistic Marker Builder.extractor")
gsm_registry = importlib.import_module("src.MAIN_PARSING.Linguistic Marker Builder.registry")
gsm_surface = importlib.import_module("src.MAIN_PARSING.Linguistic Marker Builder.surface")
from src.MAIN_PARSING.ETHICS.gender_lm_v2.surfaces import build_surfaces
from src.MAIN_PARSING.ETHICS.gender_lm_v2.detectors import detect_markers
from src.MAIN_PARSING.ETHICS.gender_lm_v2.statistics import _effect_summary, _bootstrap_matrix, _holm
REGISTRY_PATH = ROOT / "result/ETHICS/reports/gender_lm_v2_candidate_20260916_r3/registry.json"
REGISTRY = json.loads(REGISTRY_PATH.read_text(encoding="utf-8")) if BENCHMARK == "ETHICS" else [dataclasses.asdict(m) for m in gsm_registry.MARKERS]
MARKER_IDS = [m["marker_id"] for m in REGISTRY]
REGISTRY_BY_ID = {m["marker_id"]: m for m in REGISTRY}
(RUN / "registry.json").write_text(json.dumps(REGISTRY, ensure_ascii=False, indent=2), encoding="utf-8")
COMMON_IDS = [m.marker_id for m in gsm_registry.MARKERS if m.analysis_role == "primary" and "ethics" in m.benchmarks]
LABELS = dict(zip(
    ["organization_explicit_step", "organization_bullet", "organization_section_header", "stance_hedge", "stance_booster", "stance_assumption", "stance_self_correction", "connective_causal", "connective_contrastive", "connective_conditional", "self_reference_first_person", "self_reference_inclusive_we", "engagement_reader_directive", "elaboration_code_gloss", "ai_identity_statement", "stance_contextual_hedge", "stance_distancing", "connective_additive", "connective_concessive", "affect_empathy_validation", "framing_deontological", "framing_consequentialist", "framing_virtue_ethics", "framing_religious", "framing_cultural", "moral_evaluation_positive", "moral_evaluation_negative", "word_count", "avg_sentence_length", "ttr_300"],
    ["Langkah eksplisit", "Bullet", "Judul bagian", "Hedge", "Booster", "Asumsi eksplisit", "Koreksi diri", "Konektif kausal", "Konektif kontras", "Konektif kondisional", "Rujukan orang pertama", "We inklusif", "Direktif pembaca", "Penjelas ulang", "Identitas AI", "Hedge kontekstual", "Distancing", "Konektif aditif", "Konektif konsesif", "Empati / validasi", "Framing deontologis", "Framing konsekuensialis", "Framing virtue ethics", "Framing keagamaan", "Framing budaya", "Evaluasi moral positif", "Evaluasi moral negatif", "Jumlah kata", "Rata-rata kata per kalimat", "TTR hingga 300 kata"]
))
def label(marker):
    return LABELS.get(marker, marker.replace("gender_", "").replace("_", " "))
def sha(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()
def normalized(value):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", value or "")).strip()
def save_table(frame, name):
    frame.to_csv(RUN / name, index=False, encoding="utf-8-sig")
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 12, "axes.titlesize": 14, "axes.labelsize": 12, "xtick.labelsize": 11, "ytick.labelsize": 11, "figure.dpi": 110, "savefig.dpi": 180})
print("Benchmark:", BENCHMARK, "| Seluruh data mentah enam model | Output:", RUN)


## IMPLEMENTATION · Pembacaan dan pengukuran

Fungsi di bawah hanya membaca respons untuk pengukuran. Ground truth, status committed, dan correctness tidak menjadi input penghitungan marker. Aturan khusus etika tetap berstatus kandidat; definisi distancing dan overlap kontras/konsesif belum dipromosikan sebagai tervalidasi.


In [ ]:
def final_channel(obj):
    """Select response fields only; never recover empty content from reasoning."""
    response = obj.get("response") or {}
    raw = response.get("raw_api_response") or {}
    choices = raw.get("choices") if isinstance(raw, dict) else None
    message = choices[0].get("message", {}) if choices and isinstance(choices[0], dict) else {}
    reasoning = message.get("reasoning") or ""
    reasoning = reasoning if isinstance(reasoning, str) else ""
    if "content" in message:
        value, source = message["content"], "response.raw_api_response.choices[0].message.content"
    elif "content" in response:
        value, source = response["content"], "response.content_without_api_message_content"
    else:
        value, source = "", "no_final_content_field"
    status = "ok"
    if value is not None and not isinstance(value, str):
        value, status = "", "unsupported_content_type"
    content = value or ""
    if response.get("success") is False:
        content, status = "", "api_failed"
    wrapper = response.get("content")
    wrapper_diff = isinstance(wrapper, str) and normalized(wrapper) != normalized(value if isinstance(value, str) else "")
    return content, source, status, len(reasoning), sha(normalized(reasoning)) if reasoning else "", wrapper_diff

def normalize_channel_text(text):
    """Reuse audited-shape surface handling for tagged think/Harmony content."""
    channel_surface = build_surfaces(text)
    if channel_surface.status == "unknown_surface":
        return "", "unknown_surface", ";".join(channel_surface.flags)
    return channel_surface.visible_text, "", ";".join(channel_surface.flags)

def measure(content):
    """Keep each benchmark's existing marker rules; no GT or answer scoring."""
    visible, override, channel_flags = normalize_channel_text(content)
    if BENCHMARK == "GSM8K":
        values = gsm_extractor.extract_markers({"benchmark": "gsm8k", "response_content": visible})
        values.pop("response_content")
        values.pop("benchmark")
        if override:
            values["text_status"] = override
        values["surface_flags"] = channel_flags
        return values
    surface = build_surfaces(visible)
    hits = detect_markers(surface, REGISTRY)
    counts = Counter(h.marker_id for h in hits)
    available = surface.status == "substantive_explanation" and not override
    words = gsm_surface.tokenize_words(surface.explanation_text) if available else []
    alpha = [w.casefold() for w in words if any(c.isalpha() for c in w)][:300]
    sentences = max(1, gsm_surface._sentence_count(surface.explanation_text)) if available else None
    values = {
        "visible_text": surface.visible_text, "explanation_text": surface.explanation_text,
        "text_status": override or surface.status, "text_available": available,
        "word_count": surface.word_count if available else None,
        "avg_sentence_length": surface.word_count / sentences if available else None,
        "ttr_300": len(set(alpha)) / len(alpha) if alpha else None,
        "surface_flags": ";".join(filter(None, [channel_flags, *surface.flags])),
    }
    for marker in MARKER_IDS:
        count = counts[marker] if available else None
        values[f"{marker}__count"] = count
        values[f"{marker}__density"] = count * 100 / surface.word_count if available and surface.word_count else None
        values[f"{marker}__present"] = int(count > 0) if count is not None else None
    return values

CONTRASTS = []
for c in CONDITIONS:
    CONTRASTS.append((f"man_{c}", f"woman_{c}", "direct", c))
    for gender in ("man", "woman"):
        CONTRASTS.append((f"{gender}_{c}", "neutral", f"{gender}_vs_neutral", c))
for gender in ("man", "woman"):
    for c in CONDITIONS[1:]:
        CONTRASTS.append((f"{gender}_{c}", f"{gender}_explicit_1", f"{gender}_vs_explicit", c))

def raw_paths(model, persona):
    if BENCHMARK == "GSM8K":
        return sorted((ROOT / "result/GMS8K/Gender_Persona" / model / persona).glob("gsm8k_*.json"))
    base = ROOT / "result/ETHICS/v9 [THIS IS THE MAIN]"
    if persona != "neutral":
        return sorted((base / "Gender Persona" / model / persona).glob("ethics_*.json"))
    folder = base / "Neutral Persona" / model
    return sorted((folder / "neutral").glob("ethics_*.json")) if (folder / "neutral").is_dir() else sorted(folder.glob("ethics_*.json"))

if BENCHMARK == "GSM8K":
    source_questions = json.loads((ROOT / "data/gsm8k_test.json").read_text(encoding="utf-8"))
    DATASET_KEYS = {sha(normalized(r["question"])): f"gsm8k_test_{i:05d}" for i, r in enumerate(source_questions)}
    assert len(DATASET_KEYS) == len(source_questions), "Ambiguous canonical questions."
else:
    source_questions = json.loads((ROOT / "data/ETHICS_test_4subsets_10percent_from200.json").read_text(encoding="utf-8"))
    DATASET_KEYS = None

def read_record(task):
    model, persona, path = task
    binary = path.read_bytes()
    source_error = ""
    try:
        obj = json.loads(binary)
        if not isinstance(obj, dict):
            raise ValueError("Top-level JSON is not an object.")
    except (ValueError, UnicodeError) as exc:
        obj = {}
        source_error = type(exc).__name__ + ": " + str(exc)
    question = obj.get("input", {}).get("question", "") if BENCHMARK == "GSM8K" else obj.get("scenario", "")
    if not isinstance(question, str):
        question = ""
    subset = "gsm8k" if BENCHMARK == "GSM8K" else (obj.get("subset") or path.stem.split("_")[1])
    if BENCHMARK == "ETHICS" and subset not in ("commonsense", "deontology", "justice", "virtue"):
        raise ValueError(f"Unexpected subset: {path}")
    qhash = sha(normalized(question)) if normalized(question) else ""
    original_id = path.stem
    pair_key = (DATASET_KEYS.get(qhash, "") if BENCHMARK == "GSM8K" else f"{original_id}:{qhash}") if qhash else ""
    content, source, source_status, reasoning_chars, reasoning_hash, wrapper_diff = final_channel(obj)
    values = measure(content)
    if source_error:
        source, source_status = "invalid_json", "invalid_json"
    values.update({
        "model_id": model, "persona": persona, "subset": subset,
        "question_id": pair_key, "original_question_id": original_id, "question_hash": qhash,
        "question": question, "raw_source_file": str(path.relative_to(ROOT)),
        "raw_sha256": hashlib.sha256(binary).hexdigest(),
        "raw_final_text": content, "final_content_chars": len(content),
        "final_text_hash": sha(normalized(content)), "reasoning_chars": reasoning_chars,
        "reasoning_hash": reasoning_hash, "text_source": source,
        "source_status": source_status, "source_error": source_error, "wrapper_differs_from_api": wrapper_diff,
        "canonical_question_found": bool(pair_key),
        "reasoning_used": False, "scientific_status": SCIENTIFIC_STATUS,
    })
    return values

# Regression cases use the real JSON fields observed in the initial inspection.
assert final_channel({"response": {"content": "WRAPPER", "raw_api_response": {"choices": [{"message": {"content": "", "reasoning": "HIDDEN"}}]}}})[0] == ""
assert final_channel({"response": {"raw_api_response": {"choices": [{"message": {"content": "Visible", "reasoning": "HIDDEN"}}]}}})[0] == "Visible"
assert final_channel({"response": {"success": False, "content": "partial"}})[0] == ""
assert normalize_channel_text("<think>hidden</think>Visible")[0].strip() == "Visible"
assert normalize_channel_text("<think>unfinished")[1] == "unknown_surface"
assert measure("")["word_count"] is None
assert _effect_summary([2, 3], [1, 1], "density")["matched_rank_biserial"] == 1
assert _effect_summary([1, 1], [2, 3], "density")["matched_rank_biserial"] == -1
assert np.allclose(_holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])
print("Uji regresi seleksi kanal, NA, arah efek, dan Holm: lolos.")


In [ ]:
tasks = [(model, persona, path) for model in MODELS for persona in PERSONAS for path in raw_paths(model, persona)]
assert tasks, "No raw JSON found."
print(f"Memproses {len(tasks):,} JSON; tanpa filter committed/correct.")
records = []
from joblib import Parallel, delayed
with Parallel(n_jobs=4, backend="loky", return_as="generator", batch_size=128, pre_dispatch=8) as pool:
    for i, row in enumerate(pool(delayed(read_record)(task) for task in tasks), 1):
        records.append(row)
        if i % 1000 == 0 or i == len(tasks):
            print(f"{BENCHMARK}: {i:,}/{len(tasks):,}", flush=True)
records = pd.DataFrame(records)
keys = ["model_id", "subset", "persona", "question_id"]
records["duplicate_pair_key"] = records.duplicated(keys, keep=False) & records.canonical_question_found
records["pair_eligible"] = records.canonical_question_found & ~records.duplicate_pair_key
records["explanation_available"] = records.text_available.eq(True)
records["reasoning_only"] = records.final_content_chars.eq(0) & records.reasoning_chars.gt(0)
assert len(records) == len(tasks)
assert not records.reasoning_used.any()
assert not records.loc[records.reasoning_only, "explanation_available"].any()
assert records.loc[~records.explanation_available, "word_count"].isna().all()
assert not records.loc[records.pair_eligible].duplicated(keys).any()
save_table(records, "response_records.csv.gz")
audit_columns = ["model_id", "persona", "subset", "original_question_id", "question_id", "question_hash", "raw_source_file", "raw_sha256", "text_source", "source_status", "source_error", "final_content_chars", "reasoning_chars", "reasoning_only", "wrapper_differs_from_api", "canonical_question_found", "duplicate_pair_key", "pair_eligible", "text_status", "explanation_available"]
save_table(records[audit_columns], "source_audit.csv")
coverage = records.groupby(["model_id", "subset", "persona"], as_index=False).agg(
    total=("raw_source_file", "size"), available=("explanation_available", "sum"),
    reasoning_only=("reasoning_only", "sum"), pair_eligible=("pair_eligible", "sum"),
    wrapper_difference=("wrapper_differs_from_api", "sum"))
coverage["available_pct"] = 100 * coverage.available / coverage.total
save_table(coverage, "coverage.csv")
save_table(records.groupby(["model_id", "subset", "persona", "text_status"], as_index=False).size(), "text_status_counts.csv")
pair_audit = []
for (model, subset), group in records.groupby(["model_id", "subset"], sort=False):
    for left, right, role, condition in CONTRASTS:
        a = group[group.persona.eq(left)]
        b = group[group.persona.eq(right)]
        old = a[["original_question_id", "question_hash"]].merge(b[["original_question_id", "question_hash"]], on="original_question_id", suffixes=("_left", "_right"), validate="one_to_one")
        new = a[a.pair_eligible][["question_id", "question_hash", "explanation_available"]].merge(
            b[b.pair_eligible][["question_id", "question_hash", "explanation_available"]],
            on="question_id", suffixes=("_left", "_right"), validate="one_to_one")
        assert new.question_hash_left.eq(new.question_hash_right).all()
        pair_audit.append(dict(model_id=model, subset=subset, left=left, right=right, role=role, condition=condition,
                               old_id_pairs=len(old), old_id_question_mismatch=int(old.question_hash_left.ne(old.question_hash_right).sum()),
                               verified_question_pairs=len(new), paired_explanations=int((new.explanation_available_left & new.explanation_available_right).sum())))
pair_audit = pd.DataFrame(pair_audit)
save_table(pair_audit, "pairing_audit.csv")
if BENCHMARK == "GSM8K":
    previous_path = ROOT / "result/linguistic_markers/gender_gsm8k_full_source_aligned_v1_2_1_20260817/response_records.csv"
    previous = pd.read_csv(previous_path, usecols=["model_id", "persona", "question_id", "visible_text"], keep_default_na=False).rename(columns={"question_id": "original_question_id"})
    previous["old_visible_hash"] = previous.visible_text.map(lambda s: sha(normalized(s)) if normalized(s) else "")
    comparison = previous.drop(columns="visible_text").merge(records[audit_columns + ["final_text_hash", "reasoning_hash"]], on=["model_id", "persona", "original_question_id"], how="outer", indicator=True, validate="one_to_one")
    comparison["old_equals_reasoning"] = comparison.old_visible_hash.notna() & comparison.old_visible_hash.ne("") & comparison.old_visible_hash.eq(comparison.reasoning_hash)
    comparison["old_equals_current_final"] = comparison.old_visible_hash.eq(comparison.final_text_hash)
    save_table(comparison, "previous_source_comparison.csv")
    print("Teks lama cocok persis setelah normalisasi spasi dengan reasoning:", int(comparison.old_equals_reasoning.sum()))
print("Pasangan dengan nomor file sama tetapi isi soal berbeda:", int(pair_audit.old_id_question_mismatch.sum()), "(terhitung per kontras)")
display(coverage.groupby("model_id")[["total", "available", "reasoning_only"]].sum())


## CHECKLIST · Statistik kandidat

Pasangan diuji kesamaan soalnya terlebih dahulu. Wilcoxon tidak diganti sign test secara otomatis. Sign test tersedia sebagai sensitivitas; Holm diterapkan pada seluruh outcome × 18 kontras per model dan subset. Ini kebijakan rebuild setelah inspeksi, **bukan preregistrasi**.

Ukuran efek: matched rank-biserial. Selisih rata-rata, median, IQR, dan CI bootstrap 95% untuk **selisih rata-rata** disimpan terpisah. NA tidak diubah menjadi nol; rank-biserial tidak terdefinisi bila seluruh pasangan bernilai sama. Empat subset ETHICS tidak dipool. Sensitivitas correctness ditunda sampai lineage jawaban tervalidasi.


In [ ]:
# Candidate inferential outputs are explicitly not a final thesis release.
# Each outcome has one primary measurement; presence is retained separately in profiles.
OUTCOMES = [("word_count", "word_count", "count"), ("avg_sentence_length", "avg_sentence_length", "count"), ("ttr_300", "ttr_300", "count")]
OUTCOMES += [(m["marker_id"], m["marker_id"] + ("__present" if m["measurement"] == "binary" else "__density"), "presence" if m["measurement"] == "binary" else "density") for m in REGISTRY]
rows, omnibus_rows, profiles = [], [], []
rng = np.random.default_rng(SEED)
for (model, subset), group in records.groupby(["model_id", "subset"], sort=False):
    print("Statistik kandidat:", MODEL_LABELS[model], subset, flush=True)
    usable = group[group.pair_eligible]
    for persona, part in group.groupby("persona"):
        for marker in MARKER_IDS:
            counts = part[f"{marker}__count"].dropna()
            density = part[f"{marker}__density"].dropna()
            profiles.append(dict(model_id=model, subset=subset, persona=persona, marker_id=marker,
                                 total=len(part), n_available=len(counts), raw_count_sum=counts.sum() if len(counts) else np.nan,
                                 prevalence_pct=100 * counts.gt(0).mean() if len(counts) else np.nan,
                                 mean_density=density.mean(), median_density=density.median(),
                                 rule_status="candidate", theoretical_source=REGISTRY_BY_ID[marker]["theoretical_source"]))
    for left, right, role, condition in CONTRASTS:
        cols = ["question_id"] + [col for _, col, _ in OUTCOMES]
        paired = usable[usable.persona.eq(left)][cols].merge(usable[usable.persona.eq(right)][cols], on="question_id", suffixes=("_left", "_right"), validate="one_to_one")
        jobs = defaultdict(list)
        for marker, col, measurement in OUTCOMES:
            av = pd.to_numeric(paired[col + "_left"], errors="coerce").to_numpy(float)
            bv = pd.to_numeric(paired[col + "_right"], errors="coerce").to_numpy(float)
            mask = np.isfinite(av) & np.isfinite(bv)
            a, b = av[mask], bv[mask]
            result = _effect_summary(a, b, measurement)
            # Average length/TTR are not counts; correct their units explicitly.
            if marker in ("word_count", "avg_sentence_length", "ttr_300"):
                result["measurement_unit"] = {"word_count": "words", "avg_sentence_length": "words_per_sentence", "ttr_300": "ratio"}[marker]
            factor = 100 if measurement == "presence" else 1
            delta = (a - b) * factor
            result.update(model_id=model, subset=subset, left=left, right=right, role=role, condition=condition,
                          marker_id=marker, measurement=measurement, n_question_pairs=len(paired),
                          median_delta=float(np.median(delta)) if len(delta) else np.nan,
                          q25_delta=float(np.quantile(delta, .25)) if len(delta) else np.nan,
                          q75_delta=float(np.quantile(delta, .75)) if len(delta) else np.nan,
                          bootstrap_resamples=BOOTSTRAP_RESAMPLES, scientific_status=SCIENTIFIC_STATUS)
            rows.append(result)
            if len(delta):
                jobs[mask.tobytes()].append((result, delta))
        # Share question resamples only across outcomes with identical complete pairs.
        for bundle in jobs.values():
            matrix = np.column_stack([delta for _, delta in bundle])
            if np.any(matrix):
                bounds = _bootstrap_matrix(matrix, np.repeat(subset, len(matrix)), BOOTSTRAP_RESAMPLES, rng)
            else:
                bounds = np.zeros((2, len(bundle)))
            for index, (result, _) in enumerate(bundle):
                result.update(ci_low=float(bounds[0, index]), ci_high=float(bounds[1, index]),
                              ci_estimand="mean_paired_difference_pp" if result["measurement"] == "presence" else "mean_paired_difference",
                              ci_method="paired_percentile_bootstrap", ci_simultaneous=False)
    for gender in ("man", "woman"):
        personas = ["neutral"] + [f"{gender}_{c}" for c in CONDITIONS]
        for marker, col, measurement in OUTCOMES:
            if measurement == "presence":
                continue  # Binary omnibus is not silently treated as Friedman.
            wide = usable.pivot(index="question_id", columns="persona", values=col).reindex(columns=personas).dropna()
            if len(wide) < 2:
                stat, p = np.nan, np.nan
            elif np.all(np.ptp(wide.to_numpy(float), axis=1) == 0):
                stat, p = 0.0, 1.0
            else:
                test = friedmanchisquare(*[wide[p].to_numpy(float) for p in personas])
                stat, p = float(test.statistic), float(test.pvalue)
            omnibus_rows.append(dict(model_id=model, subset=subset, gender=gender, marker_id=marker,
                                      n_complete_questions=len(wide), statistic=stat, raw_p=p,
                                      kendall_w=stat/(len(wide)*4) if len(wide) else np.nan,
                                      scientific_status=SCIENTIFIC_STATUS))
comparisons = pd.DataFrame(rows)
# Fixed prospective family for THIS rebuild (not claimed to be preregistered):
# all 18 planned contrasts and all outcome markers, per model x subset.
comparisons["family_id"] = comparisons.model_id + "|" + comparisons.subset + "|all_planned_outcomes_and_contrasts"
comparisons["p_holm"] = comparisons.groupby("family_id").raw_p.transform(lambda p: _holm(p.to_numpy()))
comparisons["sign_p_holm"] = comparisons.groupby("family_id").sign_raw_p.transform(lambda p: _holm(p.to_numpy()))
comparisons["family_size"] = comparisons.groupby("family_id").marker_id.transform("size")
omnibus = pd.DataFrame(omnibus_rows)
omnibus["p_holm"] = omnibus.groupby(["model_id", "subset", "gender"]).raw_p.transform(lambda p: _holm(p.to_numpy()))
profiles = pd.DataFrame(profiles)
save_table(comparisons, "paired_comparisons_candidate.csv")
save_table(omnibus, "friedman_candidate.csv")
save_table(profiles, "marker_profiles.csv")
save_table(pd.DataFrame(REGISTRY).drop(columns=["patterns"]), "marker_registry_and_sources.csv")
assert comparisons.n_pairs.le(comparisons.n_question_pairs).all()
assert comparisons.loc[comparisons.n_pairs.eq(0), "matched_rank_biserial"].isna().all()
assert comparisons.matched_rank_biserial.dropna().between(-1, 1).all()
assert comparisons.p_holm.dropna().between(0, 1).all()
assert comparisons.loc[comparisons.raw_p.notna(), "p_holm"].ge(comparisons.loc[comparisons.raw_p.notna(), "raw_p"] - 1e-12).all()
assert len(comparisons) == records.groupby(["model_id", "subset"]).ngroups * len(CONTRASTS) * len(OUTCOMES)
print("Uji pasangan, NA, batas ukuran efek, Holm, dan kelengkapan kontras: lolos.")


## TESTING · Visualisasi PNG

Semua titik berwarna penuh. Empat panel E/I1/I2/I3 disusun 2 × 2; label dan warna model konsisten. Arah langsung **Man − Woman**. Silang di kolom NA berarti effect size tidak terdefinisi; alasannya tercatat di CSV. Segitiga berarti <20 pasangan nonnol, bukan otomatis tidak sah. Bingkai hitam berarti p Holm kandidat <0,05. Penjelasan ini berada di notebook/README agar gambar tidak dipenuhi catatan panjang.


In [ ]:
FIGURES = []
def finish(fig, name, title, table):
    fig.savefig(STAGED_FIGURES / name, facecolor="white", bbox_inches="tight")
    plt.close(fig)
    FIGURES.append(dict(figure=name, title=title, source_table=table, scientific_status=SCIENTIFIC_STATUS))
def clean(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", color="#E2E8F0", linewidth=.7)
    ax.set_axisbelow(True)
def model_legend(fig):
    handles = [Line2D([], [], linestyle="", marker="o", markersize=7, color=COLORS[m], label=MODEL_LABELS[m]) for m in MODELS]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=10.5, bbox_to_anchor=(.56, .005))

SUBSETS = sorted(records.subset.unique())
for subset in SUBSETS:
    cov = coverage[coverage.subset.eq(subset)]
    pivot = cov.pivot(index="model_id", columns="persona", values="available_pct").reindex(index=MODELS, columns=PERSONAS)
    counts = cov.pivot(index="model_id", columns="persona", values="available").reindex(index=MODELS, columns=PERSONAS)
    totals = cov.pivot(index="model_id", columns="persona", values="total").reindex(index=MODELS, columns=PERSONAS)
    fig, ax = plt.subplots(figsize=(14, 5.6))
    image = ax.imshow(pivot, vmin=0, vmax=100, cmap="Blues", aspect="auto")
    for y in range(len(MODELS)):
        for x in range(len(PERSONAS)):
            value = pivot.iloc[y, x]
            text = f"{value:.1f}%\n{counts.iloc[y,x]:.0f}/{totals.iloc[y,x]:.0f}" if np.isfinite(value) else "NA"
            ax.text(x, y, text, ha="center", va="center", fontsize=10, color="white" if value > 60 else "#172033")
    ax.set_xticks(range(9), PERSONA_LABELS, rotation=35, ha="right")
    ax.set_yticks(range(6), [MODEL_LABELS[m] for m in MODELS])
    title = f"{BENCHMARK} · {subset}: ketersediaan penjelasan"
    ax.set_title(title, pad=14, fontweight="bold")
    fig.colorbar(image, ax=ax, label="Penjelasan tersedia (%)", fraction=.025, pad=.02)
    fig.tight_layout()
    finish(fig, f"00_cakupan_{subset}.png", title, "coverage.csv")

    fig, axes = plt.subplots(2, 3, figsize=(14, 8.5), sharey=True)
    for ax, model in zip(axes.flat, MODELS):
        part = records[records.model_id.eq(model) & records.subset.eq(subset)]
        for y, persona in enumerate(PERSONAS):
            values = part.loc[part.persona.eq(persona), "word_count"].dropna()
            if len(values):
                lo, med, hi = values.quantile([.25, .5, .75])
                ax.plot([lo, hi], [y, y], color=COLORS[model], lw=2)
                ax.scatter(med, y, color=COLORS[model], s=35, zorder=3)
        ax.set_title(MODEL_LABELS[model], fontweight="bold")
        ax.set_yticks(range(9), PERSONA_LABELS)
        ax.invert_yaxis() if not ax.yaxis_inverted() else None
        ax.set_xlabel("Jumlah kata: median dan IQR")
        clean(ax)
    title = f"{BENCHMARK} · {subset}: panjang penjelasan"
    fig.suptitle(title, fontsize=18, fontweight="bold")
    fig.tight_layout(rect=(0, 0, 1, .95))
    finish(fig, f"01_panjang_penjelasan_{subset}.png", title, "response_records.csv.gz")

    families = [("umum", [m for m in COMMON_IDS if m in MARKER_IDS])]
    if BENCHMARK == "ETHICS":
        families += [
            ("etika", [m for m in MARKER_IDS if m.startswith(("framing_", "moral_evaluation_")) or m in ("stance_contextual_hedge", "stance_distancing", "connective_additive", "connective_concessive", "affect_empathy_validation")]),
            ("gender", [m for m in MARKER_IDS if m.startswith("gender_")]),
            ("respons", [m for m in MARKER_IDS if m.startswith("response_stance_") or m == "ai_identity_statement"]),
        ]
    else:
        families += [("format", [m for m in MARKER_IDS if m not in COMMON_IDS])]

    for family, markers in families:
        if not markers:
            continue
        # One model per row-offset, four conditions in a readable 2 x 2 layout.
        fig, axes = plt.subplots(2, 2, figsize=(14, max(8, len(markers)*.70 + 3)), sharex=True, sharey=True)
        for ax, c in zip(axes.flat, CONDITIONS):
            selected = comparisons[(comparisons.subset.eq(subset)) & comparisons.role.eq("direct") & comparisons.condition.eq(c)]
            for j, model in enumerate(MODELS):
                values = selected[selected.model_id.eq(model)].set_index("marker_id")
                for y, marker in enumerate(markers):
                    row = values.loc[marker]
                    yy = y + (j - 2.5) * .105
                    x = row.matched_rank_biserial
                    if np.isfinite(x):
                        ax.scatter(x, yy, s=30, color=COLORS[model], alpha=1, zorder=4,
                                   edgecolors="#111827" if row.p_holm < .05 else "white",
                                   linewidths=1.0 if row.p_holm < .05 else .35,
                                   marker="o" if row.n_changed >= 20 else "^")
                    else:
                        ax.scatter(1.15, yy, marker="x", s=24, color=COLORS[model], linewidth=1.1, clip_on=False)
            ax.axvline(0, color="#64748B", lw=1)
            ax.axvspan(1.055, 1.245, color="#F1F5F9", zorder=0)
            ax.text(1.15, -.7, "NA", ha="center", fontsize=10, color="#475569")
            ax.set_title(CONDITION_LABELS[c], fontweight="bold")
            ax.set_yticks(range(len(markers)), [label(m) for m in markers])
            ax.set_ylim(len(markers)-.4, -.9)
            ax.set_xlim(-1.05, 1.25)
            ax.set_xticks([-1, -.5, 0, .5, 1])
            ax.tick_params(axis="y", labelleft=True)
            clean(ax)
        title = f"{BENCHMARK} · {subset}: penanda {family} — Man − Woman"
        fig.suptitle(title, fontsize=18, fontweight="bold", y=.985)
        fig.supxlabel("Matched rank-biserial · negatif: Woman lebih tinggi · positif: Man lebih tinggi", y=.105, fontsize=12)
        model_legend(fig)
        fig.tight_layout(rect=(0, .14, 1, .95), h_pad=2, w_pad=2.5)
        finish(fig, f"02_efek_langsung_{family}_{subset}.png", title, "paired_comparisons_candidate.csv")

        # Neutral effect matrices retain ALL markers, models and conditions without a crowded scatter.
        for gender in ("man", "woman"):
            selected = comparisons[(comparisons.subset.eq(subset)) & comparisons.role.eq(f"{gender}_vs_neutral")]
            fig, axes = plt.subplots(2, 3, figsize=(15, max(7.5, len(markers)*.58 + 2)), sharey=True)
            for ax, model in zip(axes.flat, MODELS):
                wide = selected[selected.model_id.eq(model)].pivot(index="marker_id", columns="condition", values="matched_rank_biserial").reindex(index=markers, columns=CONDITIONS)
                cmap = plt.get_cmap("RdBu_r").copy()
                cmap.set_bad("#D5DBE3")
                im = ax.imshow(wide, cmap=cmap, vmin=-1, vmax=1, aspect="auto")
                for y in range(len(markers)):
                    for x in range(4):
                        v = wide.iloc[y, x]
                        ax.text(x, y, f"{v:+.2f}" if np.isfinite(v) else "NA", ha="center", va="center", fontsize=9.5, color="white" if abs(v)>.6 else "#172033")
                ax.set_xticks(range(4), ["E", "I1", "I2", "I3"])
                ax.set_yticks(range(len(markers)), [label(m) for m in markers])
                ax.set_title(MODEL_LABELS[model], fontweight="bold")
            title = f"{BENCHMARK} · {subset}: {family} — {gender.title()} − N"
            fig.suptitle(title, fontsize=18, fontweight="bold")
            fig.tight_layout(rect=(0, .05, .93, .94), h_pad=2)
            cax = fig.add_axes([.945, .18, .015, .62])
            fig.colorbar(im, cax=cax, label="Matched rank-biserial")
            finish(fig, f"03_efek_vs_netral_{family}_{gender}_{subset}.png", title, "paired_comparisons_candidate.csv")

    # Surface measures have their own units, not a mixed density axis.
    fig, axes = plt.subplots(3, 4, figsize=(16, 10), sharey=True)
    for i, marker in enumerate(["word_count", "avg_sentence_length", "ttr_300"]):
        for j, c in enumerate(CONDITIONS):
            ax = axes[i, j]
            part = comparisons[comparisons.subset.eq(subset) & comparisons.role.eq("direct") & comparisons.condition.eq(c) & comparisons.marker_id.eq(marker)].set_index("model_id")
            for k, model in enumerate(MODELS):
                r = part.loc[model]
                if np.isfinite(r.effect):
                    ax.plot([r.ci_low, r.ci_high], [k,k], color=COLORS[model], lw=1.8)
                    ax.scatter(r.effect, k, color=COLORS[model], s=32, zorder=3)
            ax.axvline(0, color="#64748B", lw=.8)
            ax.set_yticks(range(6), [MODEL_LABELS[m] for m in MODELS])
            ax.set_xlabel(label(marker), fontsize=10)
            ax.set_title(CONDITION_LABELS[c] if i == 0 else "")
            clean(ax)
    title = f"{BENCHMARK} · {subset}: karakteristik penjelasan — Man − Woman"
    fig.suptitle(title, fontsize=18, fontweight="bold")
    fig.supxlabel("Selisih rata-rata berpasangan dan CI bootstrap 95% (marginal)", y=.015)
    fig.tight_layout(rect=(0,.04,1,.95))
    finish(fig, f"04_karakteristik_penjelasan_{subset}.png", title, "paired_comparisons_candidate.csv")

    # Absolute prevalence prevents strong ranks on very rare markers being overinterpreted.
    for page, offset in enumerate(range(0, len(MARKER_IDS), 14), 1):
        markers = MARKER_IDS[offset:offset+14]
        fig, axes = plt.subplots(2, 3, figsize=(16, 9.5), sharey=True)
        for ax, model in zip(axes.flat, MODELS):
            wide = profiles[profiles.model_id.eq(model) & profiles.subset.eq(subset)].pivot(index="marker_id", columns="persona", values="prevalence_pct").reindex(index=markers, columns=PERSONAS)
            cmap = plt.get_cmap("YlGnBu").copy()
            cmap.set_bad("#D5DBE3")
            im = ax.imshow(wide, vmin=0, vmax=100, cmap=cmap, aspect="auto")
            ax.set_yticks(range(len(markers)), [label(m) for m in markers])
            ax.set_xticks(range(9), PERSONA_LABELS, rotation=65, ha="right", fontsize=9)
            ax.set_title(MODEL_LABELS[model], fontweight="bold")
        title = f"{BENCHMARK} · {subset}: prevalensi penanda ({page})"
        fig.suptitle(title, fontsize=18, fontweight="bold")
        fig.tight_layout(rect=(0,0,.93,.95), h_pad=2)
        cax = fig.add_axes([.95,.18,.015,.62])
        fig.colorbar(im, cax=cax, label="Respons yang memuat penanda (%)")
        finish(fig, f"05_profil_penanda_{subset}_{page:02d}.png", title, "marker_profiles.csv")

figure_index = pd.DataFrame(FIGURES)
for i, row in figure_index.iterrows():
    data = (STAGED_FIGURES / row.figure).read_bytes()
    assert data.startswith(b"\x89PNG\r\n\x1a\n")
    figure_index.loc[i, "sha256"] = hashlib.sha256(data).hexdigest()
save_table(figure_index, "figure_index.csv")
print(f"{len(figure_index)} PNG selesai dirender di staging.")


## INTEGRATION · Pemasangan lokal

PNG dan hash diperiksa sebelum folder figures diperbarui. Gambar lama dipindahkan ke arsip lokal yang dapat dipulihkan. Tidak ada sinkronisasi ke naskah atau gambar Bab VI.


In [ ]:
# Install only after data invariants and staged images passed.
from PIL import Image as PILImage
for name in figure_index.figure:
    with PILImage.open(STAGED_FIGURES / name) as image:
        image.verify()
assert len(records) == len(tasks) and len(figure_index) > 0
assert len(figure_index.figure.unique()) == len(figure_index)
assert not records.reasoning_used.any()
source_files = sorted({Path(gsm_extractor.__file__), Path(gsm_registry.__file__), Path(gsm_surface.__file__),
                       ROOT / "src/MAIN_PARSING/ETHICS/gender_lm_v2/surfaces.py",
                       ROOT / "src/MAIN_PARSING/ETHICS/gender_lm_v2/detectors.py",
                       ROOT / "src/MAIN_PARSING/ETHICS/gender_lm_v2/statistics.py",
                       ROOT / "src/MAIN_PARSING/ETHICS/gender_lm_v2/_baseline/extractor.py",
                       ROOT / "src/MAIN_PARSING/ETHICS/gender_lm_v2/_baseline/registry.py"})
source_hashes = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files}
snapshot = RUN / "code_snapshot"
for path in source_files:
    target = snapshot / path.relative_to(ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, target)
shutil.copy2(ACTIVE / NOTEBOOK_NAME, RUN / NOTEBOOK_NAME)
manifest = {
    "run_id": RUN_ID, "benchmark": BENCHMARK, "scientific_status": SCIENTIFIC_STATUS,
    "raw_rows": len(records), "available_explanations": int(records.explanation_available.sum()),
    "reasoning_only_rows": int(records.reasoning_only.sum()), "hidden_reasoning_used": False,
    "unmatched_canonical_question_rows": int((~records.canonical_question_found).sum()),
    "duplicate_pair_key_rows": int(records.duplicate_pair_key.sum()),
    "pair_key": "model + subset + canonical question content (GSM8K); source task ID + scenario hash (ETHICS)",
    "old_id_question_mismatch_across_contrasts": int(pair_audit.old_id_question_mismatch.sum()),
    "bootstrap_resamples": BOOTSTRAP_RESAMPLES, "seed": SEED,
    "effect_direction": "Man - Woman; persona - neutral; implicit - explicit",
    "holm_family": "all planned outcomes x 18 contrasts, separately per model and subset",
    "ci_estimand": "mean paired difference; percentile bootstrap 95%, marginal, not simultaneous",
    "effect_size": "matched rank-biserial on nonzero differences; undefined when no nonzero differences",
    "all_responses_inventoried": True, "committed_or_correct_filter": False,
    "correctness_sensitivity": "deferred_until_answer_and_raw_source_lineage_validated",
    "marker_rules_changed": False, "source_hashes": source_hashes,
    "final_release": False, "manual_precision_recall_audit": "pending",
    "thesis_files_modified": False, "figure_count": len(figure_index), "formats": ["png"],
}
(RUN / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
notes = f"""# {BENCHMARK}: hitung ulang dari JSON asli

Status: **kandidat, belum untuk klaim final tesis**. Run: {RUN_ID}.
Notebook: {NOTEBOOK_NAME}. Seluruh kode perbaikan berada di notebook.
Data mentah, report lama di result/, dan Bab IV/VI tidak diubah.

## SPEC
Seluruh JSON yang tersedia untuk enam model dan sembilan persona, bukan sampel.
Jumlah JSON: {len(records):,}; penjelasan tersedia: {int(records.explanation_available.sum()):,}.
Reasoning tanpa content: {int(records.reasoning_only.sum()):,}; tidak dipakai sebagai penjelasan.
Seluruh respons diinventarisasi, tanpa filter committed/correct.

## DESIGN PLAN
GSM8K dicocokkan dengan teks soal pada seluruh 1.319 soal test lokal.
ETHICS dipasangkan menggunakan ID tugas dan hash skenario; empat subset tidak dilebur.
Nomor file lama dipertahankan untuk audit, bukan dianggap bukti soal identik.
Duplikasi dan soal yang tidak dapat dicocokkan tetap dalam inventaris, dikeluarkan dari pairing.

## IMPLEMENTATION
Kanal message.content menjadi sumber utama, termasuk bila kosong.
Tidak ada fallback ke reasoning atau prediction.raw. Wrapper content hanya dipakai ketika
field message.content tidak tersedia dan sumbernya dicatat.
Kamus marker lama dipertahankan, sumber/definisi pada marker_registry_and_sources.csv.
Kamus ETHICS dan GSM8K tidak diklaim identik; penamaan serupa tidak membuktikan ekuivalensi aturan.
Density per 100 kata; count dan presence tersedia per respons.
NA tidak diganti nol. Semua pasangan bernilai sama menghasilkan selisih mentah nol,
tetapi rank-biserial tanpa pasangan nonnol tidak terdefinisi (NA; status no_change).
NA juga dapat berarti tidak ada pasangan tersedia (status unavailable); alasan ada di CSV.

## CHECKLIST
- [x] Seleksi kanal, tidak memakai reasoning.
- [x] Verifikasi isi soal setiap pasangan dan penolakan duplikasi.
- [x] Uji arah Man minus Woman, batas rank-biserial, NA dan koreksi Holm.
- [x] Hasil baru di path baru; PNG lama dapat dipulihkan dari arsip.
- [ ] Audit manual precision/recall marker.
- [ ] Validasi lengkap parsing jawaban dan lineage untuk sensitivitas correctness.
- [ ] Review konstruk distancing, empati, serta overlap kontras/konsesif.
- [ ] Rilis final untuk tesis.

## TESTING
Jalankan seluruh sel dari kernel baru. Assertions menggagalkan instalasi bila invariant gagal.
Notebook menghasilkan folder run baru setiap eksekusi, tanpa menimpa report di result/.
source_audit.csv dan pairing_audit.csv memuat bukti; source JSON memiliki SHA-256.
source_hashes pada manifest merekam versi modul; salinan kode tersimpan di code_snapshot/.

## Membaca gambar dan statistik
Semua titik memakai warna penuh yang sama untuk enam model.
Man minus Woman: negatif berarti Woman lebih tinggi, positif berarti Man lebih tinggi.
Tanda silang di kolom NA: tidak ada estimasi; bukan nol.
Segitiga: kurang dari 20 pasangan dengan selisih nonnol, **bukan** otomatis hasil tak sah.
Lingkar hitam: p Holm < 0,05 pada statistik kandidat; bukan bukti kamus sudah valid.
Keluarga Holm: seluruh outcome x 18 kontras per model dan subset, ditetapkan untuk rebuild ini;
ini perubahan analisis setelah inspeksi, bukan preregistrasi.
Wilcoxon selalu menjadi uji numerik utama; sign test menjadi sensitivitas, tidak fallback.
Presence menggunakan McNemar exact. Friedman numerik memakai soal lengkap N/E/I1/I2/I3.
Friedman tidak dihitung untuk outcome biner.
CI 95% adalah selisih rata-rata berpasangan dengan {BOOTSTRAP_RESAMPLES} bootstrap,
bukan CI rank-biserial; median delta dan IQR disediakan terpisah di CSV.
Tidak ada ambang besar/kecil otomatis untuk efek yang bertumpu sedikit pasangan.
TTR memakai hingga 300 kata alfabetis pertama; panjang <300 dicakup dan dapat memengaruhi TTR.
Grafik correctness lama tidak dipakai ulang karena lineage jawaban belum tervalidasi.

## INTEGRATION
Hasil hanya dipasang ke notebook dan folder figures benchmark; tidak disalin ke ProposalTA.
Grafik lama diarsipkan untuk mencegah hasil lama bercampur dengan rebuild.
Temuan ETHICS deontology/virtue dibaca sebagai bahasa respons terhadap prompt yang terkirim,
bukan otomatis pengukuran kemampuan moral sesuai benchmark asli.
"""
(RUN / "README.md").write_text(notes, encoding="utf-8")
gallery = "<!doctype html><html lang='id'><meta charset='utf-8'><title>" + BENCHMARK + " — penanda linguistik</title><style>body{font:16px system-ui;max-width:1200px;margin:32px auto;color:#172033}img{max-width:100%;border:1px solid #ddd}section{margin:36px 0}a{color:#165ac5}</style><h1>" + BENCHMARK + " — hasil hitung ulang</h1><p>Kandidat; audit marker belum selesai. Arah langsung: Man − Woman. Bab VI tidak diubah.</p><p><a href='README.md'>Metode, NA, statistik, dan batas klaim</a> · <a href='manifest.json'>Manifest</a></p>"
for row in figure_index.itertuples():
    gallery += f"<section><h2>{html.escape(row.title)}</h2><a href='figures/{row.figure}'><img loading='lazy' src='figures/{row.figure}'></a></section>"
gallery += "</html>"
(RUN / "gallery.html").write_text(gallery, encoding="utf-8")
# Exact active targets, never result/ or thesis folders.
active_resolved = ACTIVE.resolve()
assert active_resolved.is_relative_to((ROOT / "notebooks/gender/linguistic marker").resolve())
archive = ACTIVE / "_arsip" / f"sebelum_raw_rebuild_{RUN_ID}"
archive.mkdir(parents=True, exist_ok=False)
for name in ["figures", "figure_index.csv", "gallery.html", "README.md", "manifest.json", "presentation_source.json"]:
    source = (ACTIVE / name).resolve()
    assert source.parent == active_resolved
    if source.exists():
        shutil.move(str(source), str(archive / name))
shutil.copytree(STAGED_FIGURES, ACTIVE / "figures")
for name in ["figure_index.csv", "gallery.html", "README.md", "manifest.json"]:
    shutil.copy2(RUN / name, ACTIVE / name)
(ACTIVE / "presentation_source.json").write_text(json.dumps({
    "source_run": str(RUN), "active_notebook": str(ACTIVE / NOTEBOOK_NAME),
    "active_figures": str(ACTIVE / "figures"), "previous_active_artifacts": str(archive),
    "scientific_status": SCIENTIFIC_STATUS, "thesis_files_modified": False,
}, ensure_ascii=False, indent=2), encoding="utf-8")
for row in figure_index.itertuples():
    assert hashlib.sha256((ACTIVE / "figures" / row.figure).read_bytes()).hexdigest() == row.sha256
print("INSTALASI SELESAI:", ACTIVE / "figures")
print("Grafik lama tersimpan di:", archive)
display(Markdown(f"[Buka galeri PNG](gallery.html) · [Metode dan batas klaim](README.md)"))
display(figure_index[["figure", "title"]])
display(Image(filename=str(ACTIVE / "figures" / figure_index.iloc[0].figure), width=1000))
